The libraries are considering the native databricks enviroment at 18-03-2026

In [ ]:
#%pip install databricks-feature-engineering  #install if needed
#dbutils.library.restartPython()
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql import functions as F
from datetime import datetime

Build and register the feature table

In [ ]:
fe = FeatureEngineeringClient()

# Compute RFM features from Gold layer
snapshot_date = datetime(2011, 9, 1)

gold_df = spark.table('ml1.ml_project.gold_customer_summary')

feature_df = gold_df \
    .withColumn(
        'recency_days',
        F.datediff(F.lit(snapshot_date), F.col('last_purchase_date'))
    ) \
    .withColumn(
        'frequency',
        F.col('total_orders').cast('double')
    ) \
    .withColumn(
        'monetary',
        F.col('total_revenue').cast('double')
    ) \
    .withColumn(
        'avg_order_value',
        F.col('avg_order_value').cast('double')
    ) \
    .select('customer_id', 'recency_days', 'frequency', 'monetary', 'avg_order_value')

In [ ]:
# Write feature table — primary key is customer_id
# In Free Edition, this creates a Delta table in Unity Catalog
fe.create_table(
    name='ml1.ml_project.customer_rfm_features',
    primary_keys=['customer_id'],
    df=feature_df,
    description='RFM features for customer churn prediction. Snapshot: 2011-09-01'
)

print('Feature table created and populated.')
fe.read_table(name='ml1.ml_project.customer_rfm_features').show(5)

Create the training labels

Predict whether a customer will purchase again in the next 90 days using transactional RFM features.

In [ ]:
# Label: did the customer purchase in the 90 days AFTER the snapshot?
label_window_start = '2011-09-01'
label_window_end   = '2011-12-01'


future_buyers = spark.table('ml1.ml_project.silver_transactions') \
    .filter(
        (F.col('invoice_date') >= label_window_start) &
        (F.col('invoice_date') < label_window_end)
    ) \
    .select(F.col('Customer ID').alias('customer_id')) \
    .distinct() \
    .withColumn('purchased_next_90d', F.lit(1))


# All customers in the feature table
all_customers = spark.table('ml1.ml_project.customer_rfm_features') \
    .select('customer_id')


# Left join: customers who did NOT buy get label 0
labels_df = all_customers \
    .join(future_buyers, on='customer_id', how='left') \
    .fillna(0, subset=['purchased_next_90d'])


labels_df.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('ml1.ml_project.churn_labels')


# Sanity check
labels_df.groupBy('purchased_next_90d').count().show()